# Stage 4 - Train the MODIS->VIIRS harmonizer

**NASA Space Apps 2026 - MODIS/VIIRS hotspot harmonization**

Overlap-period matching, identity + calibration baselines, the LightGBM/Tweedie harmonizer with chronological early stopping, held-out evaluation with FRP-range/season slices, and the versioned hash-verified model artifact.

| | |
|---|---|
| Stage | 4 of 6 - offline compute |
| Network required | NO |
| Runs | the committed CLI scripts (`scripts/`) - logic stays out of cells |
| Writes to | `out/harmonization/ (artifact, evaluation evidence, daily records)` |

Rules that make this notebook reproducible (docs/DEVELOPMENT.md): record the
environment below, keep every output under the working directory, never fetch
from PyPI/GitHub/Hugging Face/REST APIs during offline compute, and keep
generated weights/datasets out of Git.


## 0. Environment

In [ ]:
import sys, platform, subprocess
from pathlib import Path

# Kaggle working dir is the ONLY writable location during offline compute.
KAGGLE_WORKING = Path("/kaggle/working")
WORK_DIR = KAGGLE_WORKING if KAGGLE_WORKING.exists() else Path.cwd() / "out"
WORK_DIR.mkdir(parents=True, exist_ok=True)
REPO_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path(__file__).resolve().parents[1] if "__file__" in dir() else Path.cwd()

print("python   :", sys.version)
print("platform :", platform.platform())
print("machine  :", platform.machine())
print("repo     :", REPO_ROOT)
print("work dir :", WORK_DIR)

RECORD = {"python": sys.version.split()[0], "platform": platform.platform(), "machine": platform.machine()}


SOURCE_DIR = Path("/kaggle/input") / "firms"
if not SOURCE_DIR.exists():
    SOURCE_DIR = REPO_ROOT / "tests" / "fixtures" / "sample_aoi"  # synthetic demo
AOI = "bbox:-123.5,39.5,-120.5,42.0"          # demo AOI; adjust for the real archive
print("source dir:", SOURCE_DIR, "| AOI:", AOI)


## 1. Train (gate -> match -> split -> baselines + GBDT -> evaluate)

The harmonizer must beat both baselines on held-out chronological splits before its estimates count as evidence at all.

In [ ]:
import json, hashlib
from pathlib import Path

def run_cli(script: str, *args: str, expect: int = 0) -> subprocess.CompletedProcess:
    """Run a committed CLI script; fail the notebook loudly on bad exits."""
    cmd = [sys.executable, str(REPO_ROOT / "scripts" / script), *args]
    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=7200, cwd=str(REPO_ROOT))
    if proc.returncode != expect:
        print(proc.stdout[-4000:]); print(proc.stderr[-4000:])
        raise RuntimeError(f"{script} exited {proc.returncode}, expected {expect}")
    print(proc.stdout[-2000:])
    return proc

def sha256(path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def latest_artifact(base: Path, prefix: str) -> Path:
    dirs = sorted((p for p in base.glob(prefix + "_*") if p.is_dir()), key=lambda p: p.stat().st_mtime)
    if not dirs:
        raise FileNotFoundError(f"no {prefix}_* artifact under {base}; did the training step run?")
    return dirs[-1]


train_out = WORK_DIR / "harmonization"
proc = run_cli("train_harmonizer.py", "--source-dir", str(SOURCE_DIR),
               "--output-dir", str(train_out), "--aoi", AOI, "--seed", "2026")
harmonizer_dir = latest_artifact(train_out, "harmonizer")
print("harmonizer artifact:", harmonizer_dir)


## 2. Held-out evaluation vs mandatory baselines

In [ ]:
import json
from pathlib import Path

evaluation = json.loads((train_out / "harmonization_evaluation.json").read_text())
for split, reports in evaluation.items():
    if reports.get("skipped"):
        print(f"{split}: skipped ({reports.get('reason')})")
        continue
    row = " | ".join(f"{name}={report.get('tweedie_deviance'):.3f}" for name, report in reports.items())
    print(f"{split}: {row}")

best = all(
    evaluation[s]["lightgbm_tweedie"]["tweedie_deviance"]
    <= min(r["tweedie_deviance"] for k, r in evaluation[s].items() if k != "lightgbm_tweedie")
    for s in ("validation", "test") if s in evaluation and not evaluation[s].get("skipped")
)
print("learned model beats both baselines on every held-out split:", best)


## 3. Inspect matched calibration records

Every matched row carries tolerances, eligibility flags and the target definition; ineligible (`no_viirs_coverage`) matches are excluded from training and never treated as zero fire activity.

In [ ]:
import pandas as pd
matched = pd.read_csv(train_out / "matched_records.csv")
print(f"{len(matched)} matched records; eligible:", int(matched["eligible"].sum()))
print(matched["ineligibility_reason"].value_counts(dropna=False).to_dict())
display_cols = [c for c in ("match_id", "modis_frp_mw", "viirs_frp_sum_mw", "viirs_matched_count", "eligible") if c in matched.columns]
print(matched[display_cols].head())
